In [ ]:
pip install -r requirements.txt

In [131]:
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns

In [132]:
# balancear las clases 
# despues los promedios
# ordenar la nb y poner cosas en funciones

In [133]:
train = pd.read_csv(r'./datasets/train.csv')
test = pd.read_csv(r'./datasets/test.csv')

In [ ]:
train.head()

In [ ]:
test.head()

In [136]:
df = pd.concat([train, test], axis=0)

In [137]:
df_final = df.copy()

In [ ]:
df_final

In [ ]:
df_final['id'].nunique()

In [140]:
df_final.to_csv('./datasets/df_final.csv', sep=';', index=False)

In [141]:
df = pd.read_csv(r'./datasets/df_final.csv', sep=';')

In [ ]:
df

## FUNCIONES

In [143]:
pd.set_option('display.max_columns', None)

In [ ]:
df.shape

In [ ]:
df.sample(5)

In [ ]:
df.columns

In [ ]:
df['satisfaction'].value_counts()

In [148]:
df['satisfaction']=df['satisfaction'].replace('neutral or dissatisfied', 'dissatisfied')

In [ ]:
df.head()

In [ ]:
df['satisfaction'].value_counts()

In [ ]:
df.dtypes

In [ ]:
df.select_dtypes(include=['object', 'category']).columns

In [ ]:
df.select_dtypes(include=['int']).columns

In [ ]:
df.select_dtypes(include=['float']).columns

In [ ]:
df.isnull().sum()

In [ ]:
df[(df['Arrival Delay in Minutes'].isna())&(df['satisfaction']=='dissatisfied')]

In [ ]:
df[(df['Arrival Delay in Minutes'].isna())&(df['satisfaction']=='satisfied')]

In [158]:
promedio_demora_satisfechos = df[df['satisfaction']=='satisfied']['Arrival Delay in Minutes'].mean().round(2)

In [ ]:
promedio_demora_satisfechos

In [160]:
# Rellenar NaN solo para los satisfechos
df.loc[(df['satisfaction'] == 'satisfied') & (df['Arrival Delay in Minutes'].isna()), 'Arrival Delay in Minutes'] = promedio_demora_satisfechos

In [ ]:
df[df['Arrival Delay in Minutes']==12.53]

In [ ]:
df[df['satisfaction']=='satisfied'].head()

In [163]:
promedio_demora_insatisfechos = df[df['satisfaction']=='dissatisfied']['Arrival Delay in Minutes'].mean().round(2)

In [ ]:
promedio_demora_insatisfechos

In [165]:
df.loc[(df['satisfaction'] == 'dissatisfied') & (df['Arrival Delay in Minutes'].isna()), 'Arrival Delay in Minutes'] = promedio_demora_insatisfechos

In [ ]:
df[df['Arrival Delay in Minutes']==17.06]

In [167]:
features = df.select_dtypes(include=['int']).columns

In [ ]:
features

In [ ]:
df.isnull().sum()

In [ ]:
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
categorical_cols

In [ ]:
df.nunique()

In [172]:
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [ ]:
df_encoded

In [174]:
bool_cols = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

In [ ]:
df.columns

In [ ]:
df_encoded.columns

In [ ]:
df_encoded.select_dtypes(include=['int']).columns

In [ ]:
df['satisfaction'].value_counts()

In [179]:
features = ['Age', 'Flight Distance', 'Inflight wifi service',
       'Departure/Arrival time convenient', 'Ease of Online booking',
       'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
       'Inflight entertainment', 'On-board service', 'Leg room service',
       'Baggage handling', 'Checkin service', 'Inflight service',
       'Cleanliness', 'Departure Delay in Minutes', 'Gender_Male',
       'Customer Type_disloyal Customer', 'Type of Travel_Personal Travel',
       'Class_Eco', 'Class_Eco Plus']

In [180]:
# Inicializar el scaler
scaler = MinMaxScaler()

# Ajustar y transformar
X_scaled = scaler.fit_transform(df_encoded[features])

# Si querés que siga siendo un DataFrame
X_scaled = pd.DataFrame(X_scaled, columns=features)

### correlation

In [192]:
columns_to_drop = ['Unnamed: 0', 'id']
df_encoded = df_encoded.drop(columns=[col for col in columns_to_drop if col in df_encoded.columns], inplace=False)

In [ ]:
df_encoded.head(3)

In [183]:
cor = df_encoded.corr()

In [ ]:
df_encoded.corr()

In [ ]:
cor.style.background_gradient(cmap='coolwarm')

In [ ]:
correlation_df = df_encoded.corr(numeric_only=True)[['satisfaction_satisfied']].drop('satisfaction_satisfied')
correlation_df = correlation_df.sort_values(by='satisfaction_satisfied', ascending=False)

plt.figure(figsize=(5, len(correlation_df) * 0.4))
sns.heatmap(correlation_df, annot=True, cmap='coolwarm', center=0)
plt.title('Correlación de features con la variable target')
plt.show()

In [ ]:
# Agregado por mi

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

In [ ]:
# 1. Definimos las variables (asegurate que satisfaction sea binaria)
X = df.drop('satisfaction', axis=1)
y = df['satisfaction'].replace('neutral or dissatisfied', 'dissatisfied')


In [ ]:
# 2. Dividimos en entrenamiento y test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
# Crear el objeto de oversampling
oversampler = RandomOverSampler(random_state=42)

# Aplicarlo solo al set de entrenamiento
X_train_over, y_train_over = oversampler.fit_resample(X_train, y_train)

# Verificar
print('Oversampling:')
print(y_train_over.value_counts())

In [ ]:
# Crear el objeto de undersampling
undersampler = RandomUnderSampler(random_state=42)

# Aplicarlo solo al set de entrenamiento
X_train_under, y_train_under = undersampler.fit_resample(X_train, y_train)

# Verificar
print('Undersampling:')
print(y_train_under.value_counts())

In [ ]:
from collections import Counter

# Función para graficar la distribución de clases
def plot_balance(y_original, y_over, y_under):
    fig, axs = plt.subplots(1, 3, figsize=(18,5))

    # Gráfico original
    axs[0].bar(Counter(y_original).keys(), Counter(y_original).values(), color='skyblue')
    axs[0].set_title('Distribución Original')
    axs[0].set_ylabel('Cantidad')
    axs[0].set_xlabel('Clases')

    # Gráfico oversampling
    axs[1].bar(Counter(y_over).keys(), Counter(y_over).values(), color='limegreen')
    axs[1].set_title('Oversampling')
    axs[1].set_ylabel('Cantidad')
    axs[1].set_xlabel('Clases')

    # Gráfico undersampling
    axs[2].bar(Counter(y_under).keys(), Counter(y_under).values(), color='tomato')
    axs[2].set_title('Undersampling')
    axs[2].set_ylabel('Cantidad')
    axs[2].set_xlabel('Clases')

    plt.tight_layout()
    plt.show()

# Llamamos a la función para graficar
plot_balance(y_train, y_train_over, y_train_under)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score
import matplotlib.pyplot as plt
import numpy as np

# One-Hot Encoding para convertir todo a números
X = pd.get_dummies(X, drop_first=True)

# 6. Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Aplicar OVERSAMPLING
oversampler = RandomOverSampler(random_state=42)
X_train_over, y_train_over = oversampler.fit_resample(X_train, y_train)

# Aplicar UNDERSAMPLING
undersampler = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = undersampler.fit_resample(X_train, y_train)

# Entrenar modelos

# Modelo con Oversampling
model_over = RandomForestClassifier(random_state=42)
model_over.fit(X_train_over, y_train_over)
y_pred_over = model_over.predict(X_test)

# Modelo con Undersampling
model_under = RandomForestClassifier(random_state=42)
model_under.fit(X_train_under, y_train_under)
y_pred_under = model_under.predict(X_test)

# 10. Calcular métricas

metrics_over = {
    'Accuracy': accuracy_score(y_test, y_pred_over),
    'Recall': recall_score(y_test, y_pred_over, pos_label='satisfied'),
    'F1 Score': f1_score(y_test, y_pred_over, pos_label='satisfied')
}

metrics_under = {
    'Accuracy': accuracy_score(y_test, y_pred_under),
    'Recall': recall_score(y_test, y_pred_under, pos_label='satisfied'),
    'F1 Score': f1_score(y_test, y_pred_under, pos_label='satisfied')
}

# Función para graficar comparación de métricas

def plot_metric_comparison(metrics_over, metrics_under):
    labels = list(metrics_over.keys())
    over_values = list(metrics_over.values())
    under_values = list(metrics_under.values())

    x = np.arange(len(labels))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10,6))
    rects1 = ax.bar(x - width/2, over_values, width, label='Oversampling', color='limegreen')
    rects2 = ax.bar(x + width/2, under_values, width, label='Undersampling', color='tomato')

    ax.set_ylabel('Valor')
    ax.set_title('Comparación de Métricas entre Oversampling y Undersampling')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()

    plt.ylim(0,1)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

#  Ejecutar los gráficos
# Comparación de métricas
plot_metric_comparison(metrics_over, metrics_under)
